<a href="https://colab.research.google.com/github/JethroTababa/JetDevFolio/blob/main/Copy_of_Beewatch_hive_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install \
fastapi \
uvicorn \
pyngrok \
nest-asyncio \
python-multipart \
librosa \
numpy \
torch \
transformers \
accelerate

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "BeeWatch Backend Running"}

In [ ]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn

In [ ]:
import threading

ngrok.set_auth_token("3G9bB2sRm8FfUh4HfroyAy6rE4L_4v1mQSdKr9STv3srxHfSh")

public_url = ngrok.connect(8000)

print(public_url)

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Run uvicorn in a separate thread
thread = threading.Thread(target=run_uvicorn)
thread.daemon = True
thread.start()

## Local Inference on GPU
Model page: https://huggingface.co/troyskie/Beewatch-hive-classifier

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/troyskie/Beewatch-hive-classifier)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [5]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("audio-classification", model="troyskie/Beewatch-hive-classifier")

INFO:     Started server process [933]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


KeyboardInterrupt: 

## Transferring the model to Hugging Face

To transfer your model to Hugging Face, you'll need to use the `huggingface_hub` library. First, ensure you have it installed and then log in to your Hugging Face account. After that, you can save your pipeline and push it to a new or existing repository on the Hugging Face Hub.

In [ ]:
# Install huggingface_hub if not already installed
!pip install huggingface_hub

In [ ]:
from huggingface_hub import notebook_login

# Log in to Hugging Face Hub
notebook_login()

Now you can save your pipeline and push it to the Hugging Face Hub. Replace `"your-username/your-repo-name"` with your desired repository name on Hugging Face.

### Connecting to the FastAPI Application API

To interact with your running FastAPI application, you'll need to send a POST request to its `/classify` endpoint. This endpoint expects an audio file. Ensure your `ngrok` tunnel and `uvicorn` server are running (from cells `yrO7vyYurepT` and `rGGU_eF0rnAZ`) to get the `public_url`.

In [ ]:
import requests
from google.colab import files

# Ensure the FastAPI app and ngrok tunnel are running
# The public_url should be available from the output of cell 'yrO7vyYurepT'

# You might need to re-run the ngrok cell if the tunnel has expired or if you restarted the runtime.
# For example, if your public_url was 'https://abcd.ngrok-free.app',
# it would be public_url = 'https://abcd.ngrok-free.app'

# Assuming public_url from cell `yrO7vyYurepT` is assigned as a string
# If not, you might need to manually copy it from the output of that cell.
# For example: public_url = 'YOUR_NGROK_PUBLIC_URL_HERE'

# Upload an audio file to send to the API
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Uploading {filename} to the API...')
    # Construct the full API endpoint URL
    api_url = f"{public_url.public_url}/classify"

    # Prepare the file for the request
    files_to_upload = {'audio_file': (filename, uploaded[filename], 'audio/wav')}

    # Send the POST request
    try:
        response = requests.post(api_url, files=files_to_upload)
        response.raise_for_status() # Raise an exception for HTTP errors

        # Print the API response
        print("API Response:")
        print(response.json())
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to the API: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
# Define the repository ID (e.g., "your-username/your-model-name")
repo_id = "your-username/your-repo-name" # IMPORTANT: Change this to your desired repo ID

# Save the pipeline and push it to the Hugging Face Hub
pipe.push_to_hub(repo_id)

In [ ]:
# Load model directly
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

extractor = AutoFeatureExtractor.from_pretrained("troyskie/Beewatch-hive-classifier")
model = AutoModelForAudioClassification.from_pretrained("troyskie/Beewatch-hive-classifier", device_map="auto")

In [ ]:
import shutil

from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware

from transformers import pipeline


In [ ]:
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [ ]:
@app.post("/classify")
async def classify(audio_file: UploadFile = File(...)):

    temp_path = f"/tmp/{audio_file.filename}"

    with open(temp_path, "wb") as buffer:
        shutil.copyfileobj(audio_file.file, buffer)

    results = pipe(temp_path)

    prediction = max(results, key=lambda x: x["score"])

    return {
        "label": prediction["label"],
        "confidence": round(prediction["score"] * 100, 2),
        "all_predictions": results
    }

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3G9bB2sRm8FfUh4HfroyAy6rE4L_4v1mQSdKr9STv3srxHfSh")

public_url = ngrok.connect(8001)

print("Backend URL:")
print(public_url.public_url)

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8001,
    log_level="info"
)

server = uvicorn.Server(config)

await server.serve()